<a href="https://colab.research.google.com/github/bertholeo/IC/blob/main/IC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# instalando bibliotacas do ESM
!pip install -q "esm@git+https://github.com/evolutionaryscale/esm.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [ ]:
import os
import time
import torch
import pandas as pd

from tqdm import tqdm
from getpass import getpass
from esm.sdk.forge import ESM3ForgeInferenceClient
from esm.sdk.api import ESMProtein, LogitsConfig

# local do arquivo csv da nossa base de dados e local da onde estarão os embeddings
csv_file = "/content/Swissprot.csv"
output_dir = "/content/embeddings/Swissprot/esmc_300m"

# nome do modelo que fará os embeddings e url do forge para uso remoto d
model_version = "esmc-300m-2024-12"
forge_url = "https://biohub.ai"

# Respectivamente, quantas proteínas queremos fazer o embedding e
# quantas proteínas queremos por batch, processadas em intervals segundos.

num_inicial = 17000
num_final = 28304
batch_size = 500
interval = 60

# verifica se a pasta que guardará os embeddings existe, senão a cria
os.makedirs(output_dir, exist_ok=True)

# recebe o token do biohub, criando o cliente.
my_token = getpass("Digite seu token: ")

forge_client = ESM3ForgeInferenceClient(
    model=model_version,
    url=forge_url,
    token=my_token
)

# vamos usar pandas para lidar com o csv
df = pd.read_csv(csv_file)

# tratamento simples que converte sequência e código da proteína pra str e limpa linhas incoerentes, cria subset
df = df.dropna(subset=["ACC", "Sequence"]).copy()
df["ACC"] = df["ACC"].astype(str)
df["Sequence"] = df["Sequence"].astype(str)

df = df[df["Sequence"].str.len() > 0].copy()
df_subset = df.iloc[num_inicial:num_final].copy()

# conta proteínas processadas, guarda as que falharam e o tempo de processamento
count = 0
start_time = time.time()
failed = []

for _, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
    count += 1
    # salva embedding pelo ACC(acession number)
    acc = row["ACC"]
    sequence = row["Sequence"]
    output_file = os.path.join(output_dir, f"{acc}.pt")

    if os.path.exists(output_file):
        continue

    try:
        protein = ESMProtein(sequence=sequence)
        protein_tensor = forge_client.encode(protein)

        logits_output = forge_client.logits(
            protein_tensor,
            LogitsConfig(sequence=True, return_embeddings=True)
        )

        representation = logits_output.embeddings.detach().cpu().squeeze(0)
        torch.save(representation, output_file)
    # trata erro
    except Exception as e:
        print(f"\nFailed on {acc}: {e}")
        failed.append((acc, str(e)))
    # conta tempo por batch
    if count % batch_size == 0:
        elapsed_time = time.time() - start_time

        if elapsed_time < interval:
            time.sleep(interval - elapsed_time)

        start_time = time.time()

# cria uma lista com arquivos salvos
saved_files = [f for f in os.listdir(output_dir) if f.endswith(".pt")]

print("Embeddings salvos em:", output_dir)
print("Número de embeddings salvos:", len(saved_files))
print("Número de proteínas falhas:", len(failed))

Digite seu token: ··········


 25%|██▌       | 2880/11303 [12:10<32:46,  4.28it/s]


Failed on Q9I7U4: Input must be an ESMProteinTensor instance, but received an ESMProteinError instead: 422 Failure in encode: "422: Input sequence length (18141) exceeds maximum allowed sequence length (16384)"


 66%|██████▌   | 7440/11303 [32:23<14:14,  4.52it/s]

In [ ]:
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.2 MB/s eta 0:00:00


In [ ]:
import os
import random
import torch
import pandas as pd
import torch.nn.functional as F

from tqdm import tqdm
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.metrics import f1_score

# base de dados original
csv_file = "/content/Swissprot.csv"

# embeddings da base de dados, já num tenson pytorch
embedding_dirs = [
]

# as 10 localizações(labels)
label_cols = [
    "Membrane",
    "Cytoplasm",
    "Nucleus",
    "Extracellular",
    "Cell membrane",
    "Mitochondrion",
    "Plastid",
    "Endoplasmic reticulum",
    "Lysosome/Vacuole",
    "Golgi apparatus",
    "Peroxisome"
]

# parâmetros para gnn
k_neighbors = 10
hidden_dim = 256
dropout = 0.3
lr = 1e-3
weight_decay = 1e-4
num_epochs = 100
threshold = 0.5

# se possível, vamos usar uma gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# para garantir reprodução, vamos fazer o split toda vez de forma diferente
seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# carregando e tratando o csv
df = pd.read_csv(csv_file)

df = df.dropna(subset=["ACC"]).copy()
df["ACC"] = df["ACC"].astype(str)

for col in label_cols:
    df[col] = df[col].fillna(0).astype(float)


embedding_paths = {}

for folder in embedding_dirs:
    for filename in os.listdir(folder):
        if filename.endswith(".pt"):
            acc = filename.replace(".pt", "")
            path = os.path.join(folder, filename)
            embedding_paths[acc] = path

print("Total de emdeddings .pt encontrados:", len(embedding_paths))

df = df[df["ACC"].isin(embedding_paths.keys())].copy()
df = df.reset_index(drop=True)

print("Proteínas com embedding e rótulo:", len(df))

# carregando, tratando e normalizando embeddings
features = []
labels = []
accs = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    acc = row["ACC"]
    path = embedding_paths[acc]

    emb = torch.load(path, map_location="cpu")
    # caso os embedding forem bidimensionais, ele ainda estão na forma aminoácido/resíduo
    # precisamos tirar a média para gerar embedding único na proteína
    if emb.ndim == 3:
        emb = emb.squeeze(0)

    if emb.ndim == 2:
        emb = emb.float().mean(dim=0)
    elif emb.ndim == 1:
        emb = emb.float()
    else:
        print("Formato estranho ignorado:", acc, emb.shape)
        continue

    y = torch.tensor(row[label_cols].values.astype(float), dtype=torch.float)

    features.append(emb)
    labels.append(y)
    accs.append(acc)

x = torch.stack(features)
y = torch.stack(labels)

print("Shape X:", x.shape)
print("Shape y:", y.shape)

x = F.normalize(x, p=2, dim=1)



# criando o grafo por similaridade cosseno, com k=10 arestas para cada proteína

def build_cosine_knn_graph(x, k=10, batch_size=1024, device="cuda"):
    x_device = x.to(device)
    num_nodes = x.size(0)

    edge_src = []
    edge_dst = []

    for start in tqdm(range(0, num_nodes, batch_size)):
        end = min(start + batch_size, num_nodes)

        sim = x_device[start:end] @ x_device.T

        row_indices = torch.arange(start, end, device=device)
        sim[torch.arange(end - start, device=device), row_indices] = -float("inf")

        topk = torch.topk(sim, k=k, dim=1).indices

        src = torch.arange(start, end, device=device).unsqueeze(1).repeat(1, k).reshape(-1)
        dst = topk.reshape(-1)

        edge_src.append(src.cpu())
        edge_dst.append(dst.cpu())

    edge_src = torch.cat(edge_src)
    edge_dst = torch.cat(edge_dst)

    edge_index = torch.stack([edge_src, edge_dst], dim=0)

    reverse_edge_index = torch.stack([edge_dst, edge_src], dim=0)
    edge_index = torch.cat([edge_index, reverse_edge_index], dim=1)

    edge_index = torch.unique(edge_index, dim=1)

    return edge_index


edge_index = build_cosine_knn_graph(
    x=x,
    k=k_neighbors,
    batch_size=1024,
    device=device
)

print("Edge index:", edge_index.shape)


#criando datasets de teste, validação e treino (15:15:70)

num_nodes = x.size(0)
indices = torch.randperm(num_nodes)

train_size = int(0.70 * num_nodes)
val_size = int(0.15 * num_nodes)

train_idx = indices[:train_size]
val_idx = indices[train_size:train_size + val_size]
test_idx = indices[train_size + val_size:]

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True


# criando o grafo no pytorch

data = Data(
    x=x,
    y=y,
    edge_index=edge_index,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

data = data.to(device)

print(data)

# nossa gnn

class SimpleGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super().__init__()

        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)

        return x


model = SimpleGNN(
    input_dim=data.x.size(1),
    hidden_dim=hidden_dim,
    output_dim=len(label_cols),
    dropout=dropout
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay
)

criterion = nn.BCEWithLogitsLoss()


# funções de treino e avaliação
def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    logits = model(data.x, data.edge_index)
    loss = criterion(logits[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate(mask):
    model.eval()

    logits = model(data.x, data.edge_index)
    loss = criterion(logits[mask], data.y[mask]).item()

    probs = torch.sigmoid(logits[mask])
    preds = (probs >= threshold).float()

    y_true = data.y[mask].detach().cpu().numpy()
    y_pred = preds.detach().cpu().numpy()

    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    sample_f1 = f1_score(y_true, y_pred, average="samples", zero_division=0)

    return {
        "loss": loss,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "sample_f1": sample_f1
    }


# treinando
best_val_loss = float("inf")
best_model_path = "/content/best_simple_gnn.pt"

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch()
    val_metrics = evaluate(data.val_mask)

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        torch.save(model.state_dict(), best_model_path)

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Micro F1: {val_metrics['micro_f1']:.4f} | "
        f"Val Macro F1: {val_metrics['macro_f1']:.4f} | "
        f"Val Sample F1: {val_metrics['sample_f1']:.4f}"
    )


#testando

model.load_state_dict(torch.load(best_model_path, map_location=device))

test_metrics = evaluate(data.test_mask)

print("\nResultados no teste:")
print("Test Loss:", test_metrics["loss"])
print("Test Micro F1:", test_metrics["micro_f1"])
print("Test Macro F1:", test_metrics["macro_f1"])
print("Test Sample F1:", test_metrics["sample_f1"])


# prevendo algumas proteínas
@torch.no_grad()
def predict_some(n=10):
    model.eval()

    logits = model(data.x, data.edge_index)
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).int()

    for i in range(min(n, len(accs))):
        predicted_labels = [
            label_cols[j]
            for j in range(len(label_cols))
            if preds[i, j].item() == 1
        ]

        real_labels = [
            label_cols[j]
            for j in range(len(label_cols))
            if data.y[i, j].item() == 1
        ]

        print("\nACC:", accs[i])
        print("Real:", real_labels)
        print("Predito:", predicted_labels)


predict_some(10)

Total de emdeddings .pt encontrados: 16999
Proteínas com embedding e rótulo: 16999


100%|██████████| 16999/16999 [02:16<00:00, 124.74it/s]



Número de embeddings carregados com sucesso: 16999
Shape X: torch.Size([16999, 960])
Shape y: torch.Size([16999, 11])


100%|██████████| 17/17 [00:00<00:00, 62.36it/s]


Edge index: torch.Size([2, 209686])
Data(x=[16999, 960], edge_index=[2, 209686], y=[16999, 11], train_mask=[16999], val_mask=[16999], test_mask=[16999])
Epoch 001 | Train Loss: 0.6932 | Val Loss: 0.6547 | Val Micro F1: 0.2496 | Val Macro F1: 0.2329 | Val Sample F1: 0.2441
Epoch 002 | Train Loss: 0.6548 | Val Loss: 0.6180 | Val Micro F1: 0.2516 | Val Macro F1: 0.2333 | Val Sample F1: 0.2479
Epoch 003 | Train Loss: 0.6181 | Val Loss: 0.5811 | Val Micro F1: 0.2901 | Val Macro F1: 0.2359 | Val Sample F1: 0.3125
Epoch 004 | Train Loss: 0.5812 | Val Loss: 0.5438 | Val Micro F1: 0.3506 | Val Macro F1: 0.2180 | Val Sample F1: 0.3712
Epoch 005 | Train Loss: 0.5439 | Val Loss: 0.5070 | Val Micro F1: 0.3676 | Val Macro F1: 0.1729 | Val Sample F1: 0.3201
Epoch 006 | Train Loss: 0.5071 | Val Loss: 0.4725 | Val Micro F1: 0.2412 | Val Macro F1: 0.0979 | Val Sample F1: 0.1532
Epoch 007 | Train Loss: 0.4726 | Val Loss: 0.4423 | Val Micro F1: 0.1033 | Val Macro F1: 0.0409 | Val Sample F1: 0.0585
Epoch 0

KeyboardInterrupt: 